# 02 — NumPy Foundations

**Context:** Pure numerical computing exercises using business-flavored scenarios. No pandas — numpy only.

**Reference:** [NumPy documentation](https://numpy.org/doc/stable/)

**Allowed libraries:** `numpy` only


In [ ]:
import numpy as np

---
## Exercise 1 — Array Creation & Inspection

**Business context:** You're modeling monthly revenue for 5 product lines over 3 years (36 months).

1. Create `revenue` — a 2D array of shape `(5, 36)` with random integer values between 10,000 and 500,000. Use `np.random.seed(42)`.
2. Create `growth_rates` — a 1D array of 36 monthly growth multipliers, evenly spaced from `0.98` to `1.05`.
3. Write a function `array_summary(arr)` that returns a dict with keys: `shape`, `dtype`, `min`, `max`, `mean`, `std` — all rounded to 2 decimals where applicable.

In [ ]:
np.random.seed(42)

# YOUR CODE HERE
revenue = None
growth_rates = None

def array_summary(arr: np.ndarray) -> dict:
    """
    Returns dict: shape, dtype, min, max, mean, std
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- ASSERTIONS ---
assert revenue.shape == (5, 36), f"revenue shape must be (5,36), got {revenue.shape}"
assert revenue.min() >= 10000 and revenue.max() <= 500000, "Values out of range"
assert growth_rates.shape == (36,), "growth_rates must have 36 elements"
assert abs(growth_rates[0] - 0.98) < 1e-6, "First growth rate must be 0.98"
assert abs(growth_rates[-1] - 1.05) < 1e-6, "Last growth rate must be 1.05"

s = array_summary(revenue)
assert set(s.keys()) == {'shape', 'dtype', 'min', 'max', 'mean', 'std'}
assert s['shape'] == (5, 36)
print("✓ Exercise 1 passed")
print(array_summary(revenue))

---
## Exercise 2 — Indexing, Slicing & Boolean Masks

**Business context:** Analyze the revenue array from Exercise 1.

1. Extract `q4_revenue`: the last 3 months (months 34–36) for all product lines. Shape must be `(5, 3)`.
2. Extract `top_product_series`: the full time series of the product line with the highest **total** revenue.
3. Create a boolean mask `high_months`: `True` for any (product, month) cell where revenue exceeds the **overall mean**.
4. Calculate `high_month_count`: a 1D array of shape `(5,)` counting how many months each product line exceeded the mean.

In [ ]:
# YOUR CODE HERE
q4_revenue = None
top_product_series = None
high_months = None
high_month_count = None

In [ ]:
# --- ASSERTIONS ---
assert q4_revenue.shape == (5, 3), "q4_revenue must be (5,3)"
assert np.array_equal(q4_revenue, revenue[:, -3:]), "q4_revenue must be last 3 months"
assert top_product_series.shape == (36,), "top_product_series must have 36 months"
assert top_product_series.sum() == revenue.sum(axis=1).max(), "Must be the highest total revenue product"
assert high_months.shape == revenue.shape and high_months.dtype == bool
assert high_month_count.shape == (5,)
assert high_month_count.sum() == high_months.sum()
print("✓ Exercise 2 passed")
print(f"High-revenue months per product: {high_month_count}")

---
## Exercise 3 — Broadcasting

**Business context:** Apply the growth rate multipliers to project future revenue WITHOUT using any loops.

1. Apply `growth_rates` to `revenue` using broadcasting: multiply each month's revenue (column) by the corresponding growth rate. Result shape: `(5, 36)`. Assign to `projected_revenue`.
2. Compute `cumulative_growth`: for each product line, the cumulative product of `growth_rates` applied — i.e., what a $1 investment grows to after each month. Shape: `(36,)`. Hint: `np.cumprod`.
3. Normalize `revenue` so that each product line has values between 0 and 1 (min-max per row). Assign to `revenue_normalized`. Shape: `(5, 36)`. No loops.

In [ ]:
# YOUR CODE HERE
projected_revenue = None
cumulative_growth = None
revenue_normalized = None

In [ ]:
# --- ASSERTIONS ---
assert projected_revenue.shape == (5, 36)
assert np.allclose(projected_revenue[0, 0], revenue[0, 0] * growth_rates[0])
assert np.allclose(projected_revenue[2, 10], revenue[2, 10] * growth_rates[10])

assert cumulative_growth.shape == (36,)
assert abs(cumulative_growth[0] - growth_rates[0]) < 1e-9

assert revenue_normalized.shape == (5, 36)
assert np.allclose(revenue_normalized.min(axis=1), 0), "Row mins must be 0"
assert np.allclose(revenue_normalized.max(axis=1), 1), "Row maxes must be 1"
print("✓ Exercise 3 passed")

---
## Exercise 4 — Linear Algebra

**Business context:** Portfolio optimization basics. You have return data for 4 assets.

1. Create a `(4, 4)` covariance matrix `cov_matrix` from a `(200, 4)` array of simulated daily returns. Use `np.random.seed(0)`, generate returns from a normal distribution with mean 0.001 and std 0.02. Use `np.cov` (note: it expects variables as rows).
2. Compute the **correlation matrix** `corr_matrix` from `cov_matrix` without using any stats library — derive it mathematically from the covariance matrix.
3. Given equal weights `w = [0.25, 0.25, 0.25, 0.25]`, compute `portfolio_variance` = `w @ cov_matrix @ w.T`.
4. Compute the **eigenvalues** of `cov_matrix`. Assign sorted descending to `eigenvalues`.

In [ ]:
np.random.seed(0)

# YOUR CODE HERE
cov_matrix = None
corr_matrix = None
portfolio_variance = None
eigenvalues = None

In [ ]:
# --- ASSERTIONS ---
assert cov_matrix.shape == (4, 4), "cov_matrix must be 4x4"
assert np.allclose(cov_matrix, cov_matrix.T), "Covariance matrix must be symmetric"

assert corr_matrix.shape == (4, 4)
assert np.allclose(np.diag(corr_matrix), 1.0), "Diagonal of correlation matrix must be 1"
assert (np.abs(corr_matrix) <= 1.0 + 1e-9).all(), "Correlations must be in [-1, 1]"

assert isinstance(portfolio_variance, float) or portfolio_variance.ndim == 0, "portfolio_variance must be scalar"
assert portfolio_variance > 0, "Variance must be positive"

assert eigenvalues.shape == (4,)
assert eigenvalues[0] >= eigenvalues[-1], "Eigenvalues must be sorted descending"
print(f"✓ Exercise 4 passed — portfolio variance: {float(portfolio_variance):.8f}")

---
## Exercise 5 — Vectorized Statistics

**Business context:** Calculate risk metrics on asset returns without any loops.

Using the `(200, 4)` returns array from Exercise 4:

1. `daily_sharpe`: Sharpe ratio per asset = `mean / std` for each of the 4 assets. Shape: `(4,)`.
2. `var_95`: Value at Risk at 95% confidence for each asset (the 5th percentile of returns). Shape: `(4,)`. Use `np.percentile`.
3. `max_drawdown`: For each asset, the maximum drawdown = `(peak - trough) / peak` where peak is the running maximum. Shape: `(4,)`. No loops — use `np.maximum.accumulate`.
4. `rolling_vol_30`: 30-day rolling standard deviation for asset 0 only. Shape: `(171,)` — one value per valid 30-day window.

In [ ]:
np.random.seed(0)
returns = np.random.normal(0.001, 0.02, size=(200, 4))

# YOUR CODE HERE
daily_sharpe = None
var_95 = None
max_drawdown = None
rolling_vol_30 = None

In [ ]:
# --- ASSERTIONS ---
assert daily_sharpe.shape == (4,)
assert np.allclose(daily_sharpe, returns.mean(axis=0) / returns.std(axis=0))

assert var_95.shape == (4,)
assert (var_95 < 0).all(), "VaR 95% should be negative (left tail)"

assert max_drawdown.shape == (4,)
assert (max_drawdown >= 0).all(), "Drawdown must be non-negative"

assert rolling_vol_30.shape == (171,), f"Expected (171,), got {rolling_vol_30.shape}"
print("✓ Exercise 5 passed")
print(f"Sharpe ratios: {daily_sharpe.round(4)}")
print(f"VaR 95%: {var_95.round(4)}")

---
## Exercise 6 — Array Manipulation & Structured Operations

**Business context:** Retail sales data reshaping and aggregation.

You have raw sales data as a flat 1D array of 1200 values representing: 4 regions × 12 months × 25 products (in that order).

1. Reshape `sales_flat` into `sales_3d` with shape `(4, 12, 25)`.
2. Compute `regional_monthly_total`: total sales per region per month. Shape: `(4, 12)`.
3. Compute `best_product_per_month`: for each month (across all regions combined), the index of the top-selling product. Shape: `(12,)`. Use `np.argmax`.
4. Compute `yoy_change`: for each region and product, the difference in total (summed over months) between the last 6 months and the first 6 months, as a percentage of the first 6 months. Shape: `(4, 25)`. No loops.

In [ ]:
np.random.seed(7)
sales_flat = np.random.randint(100, 5000, size=1200)

# YOUR CODE HERE
sales_3d = None
regional_monthly_total = None
best_product_per_month = None
yoy_change = None

In [ ]:
# --- ASSERTIONS ---
assert sales_3d.shape == (4, 12, 25)
assert sales_3d.sum() == sales_flat.sum()

assert regional_monthly_total.shape == (4, 12)
assert np.allclose(regional_monthly_total.sum(), sales_flat.sum())

assert best_product_per_month.shape == (12,)
assert best_product_per_month.dtype in [np.int32, np.int64, int]

assert yoy_change.shape == (4, 25)
print("✓ Exercise 6 passed")

---
## Exercise 7 — Random Simulation (Monte Carlo)

**Business context:** Estimate the probability that a project finishes within budget using Monte Carlo simulation.

A project has 8 independent cost components. Each follows a different distribution:
- Components 0–3: Normal(mean=50k, std=10k)
- Components 4–6: Uniform(low=30k, high=80k)
- Component 7: Log-normal (underlying normal mean=10.5, std=0.3)

Budget cap: **$420,000**. Run **100,000 simulations**.

1. Simulate all 100,000 runs in a single `(100000, 8)` array `cost_simulations`. Use `np.random.seed(99)`.
2. Compute `total_costs`: sum across components. Shape: `(100000,)`.
3. `prob_within_budget`: scalar float, proportion of simulations where total cost ≤ 420,000.
4. `p10_p50_p90`: a tuple of the 10th, 50th, 90th percentiles of `total_costs`.

In [ ]:
np.random.seed(99)
BUDGET = 420_000
N_SIMS = 100_000

# YOUR CODE HERE
cost_simulations = None
total_costs = None
prob_within_budget = None
p10_p50_p90 = None

In [ ]:
# --- ASSERTIONS ---
assert cost_simulations.shape == (100_000, 8), "Must have 100k simulations x 8 components"
assert total_costs.shape == (100_000,)
assert 0 < prob_within_budget < 1, "Probability must be between 0 and 1"
assert len(p10_p50_p90) == 3
assert p10_p50_p90[0] <= p10_p50_p90[1] <= p10_p50_p90[2], "Percentiles must be ordered"
print(f"✓ Exercise 7 passed")
print(f"P(within budget): {prob_within_budget:.2%}")
print(f"P10/P50/P90: ${p10_p50_p90[0]:,.0f} / ${p10_p50_p90[1]:,.0f} / ${p10_p50_p90[2]:,.0f}")

---
## Exercise 8 — Performance: Vectorization vs Loops

**Business context:** A legacy codebase computes customer lifetime value with a slow loop. Rewrite it.

Given:
- `monthly_revenue`: shape `(10000, 24)` — 10k customers, 24 months of revenue.
- `churn_prob`: shape `(10000,)` — monthly churn probability per customer.
- `discount_rate = 0.01` per month.

CLV formula per customer: `sum over t of revenue[t] * (1 - churn_prob)^t * (1 / (1 + discount_rate))^t`

1. Implement `clv_loop(revenue, churn, rate)` using explicit Python loops (reference baseline).
2. Implement `clv_vectorized(revenue, churn, rate)` using **only numpy operations, no loops**.
3. Both must return a `(10000,)` array of CLV per customer.
4. Results must match within `1e-6` absolute tolerance.

In [ ]:
np.random.seed(1)
monthly_revenue = np.random.uniform(50, 500, size=(10000, 24))
churn_prob = np.random.uniform(0.01, 0.15, size=(10000,))
discount_rate = 0.01

def clv_loop(revenue, churn, rate):
    """CLV using Python loops — reference implementation."""
    # YOUR CODE HERE
    pass

def clv_vectorized(revenue, churn, rate):
    """CLV using pure numpy — no loops."""
    # YOUR CODE HERE
    pass

In [ ]:
import time

clv_loop_result = clv_loop(monthly_revenue, churn_prob, discount_rate)
clv_vec_result = clv_vectorized(monthly_revenue, churn_prob, discount_rate)

assert clv_loop_result.shape == (10000,)
assert clv_vec_result.shape == (10000,)
assert np.allclose(clv_loop_result, clv_vec_result, atol=1e-6), "Results must match"

t0 = time.time(); clv_loop(monthly_revenue, churn_prob, discount_rate); loop_time = time.time() - t0
t0 = time.time(); clv_vectorized(monthly_revenue, churn_prob, discount_rate); vec_time = time.time() - t0

print(f"✓ Exercise 8 passed")
print(f"Loop time: {loop_time:.3f}s | Vectorized time: {vec_time:.3f}s | Speedup: {loop_time/vec_time:.1f}x")